# Tarrant CAD — ResidentialCompAttributeData ETL

Converts a Tarrant Appraisal District residential comp attribute workbook (`.xlsx`) to CSV.

**Source format** (checked on the 2025 and 2026 files):
- Data is on the first sheet, 18 columns, ~660K rows, one row per `PIN`
- 2026 also has a `Data Dictionary` sheet (not exported) and renamed headers vs 2025

Steps: **Load raw → Export**.

## 1. Setup

In [11]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA_DIR = Path("../Data")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

INPUT_FILE = DATA_DIR / "2026ResidentialImprovementDetails.xlsx"
OUTPUT_FILE = OUTPUT_DIR / "2026ResidentialImprovementDetails.csv"
print(INPUT_FILE.resolve(), f"{INPUT_FILE.stat().st_size / 1e6:,.1f} MB")

/Users/user2/PTP/CAD Website/Tarrant/Data/2026ResidentialImprovementDetails.xlsx 14.7 MB


## 2. Load raw
Read everything as string so nothing (leading zeros in `PIN`) gets mangled by type inference.

In [12]:
raw = pd.read_excel(
    INPUT_FILE,
    sheet_name=0,            # data sheet; 2026 also has a "Data Dictionary" sheet
    dtype=str,
    keep_default_na=False,   # keep blanks as "" instead of NaN
    engine="calamine",       # much faster than openpyxl on large files (pip install python-calamine)
)
raw.columns = raw.columns.str.strip()
print(raw.shape)
raw.head()

(692469, 4)


,TaxYear,PIN,ImprovementDetail,ImprovementValue
0,2026,00001309,Garage,10211
1,2026,00008214,Garage,4007
2,2026,00008265,Enclosed Porch,666
3,2026,00008427,Garage,2333
4,2026,00008435,Garage,7817


## 3. Export

In [13]:
raw.to_csv(OUTPUT_FILE, index=False)